# Import libraries and set configs

In [ ]:
import sys

sys.path.append("..")
sys.path.append("../..")

import ast
import json
import joblib
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb


class CFG:
    # fold - train and validate data on TSS fold scheme
    # inference - train model on all available data and save it
    train_test = "inference"
    n_folds = 5
    min_precision = 0.5
    test_time_days = 90

# Load the train data

In [18]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(90, unit="D")

profitable_hours_df = pd.read_csv("data/profitable_hours.csv")
latest = profitable_hours_df.iloc[-1]
buy_hours = ast.literal_eval(latest["profitable_buy_hours"])
sell_hours = ast.literal_eval(latest["profitable_sell_hours"])

buy_mask = (train_df["ttype"] == "buy") & (train_df["time"].dt.hour.isin(buy_hours))
sell_mask = (train_df["ttype"] == "sell") & (train_df["time"].dt.hour.isin(sell_hours))
train_df = train_df[buy_mask | sell_mask].reset_index(drop=True)

# Train

### Load best parameters from Optuna dataframe

In [19]:
from utils.optimization_utils import load_best_params

params = load_best_params(row_num=0)
params

{'boosting_type': 'dart',
 'class_weight': 'balanced',
 'colsample_bytree': 0.887251061084062,
 'corr_thresh': 0.9581562610688436,
 'feature_num': 63,
 'high_bound': 0.5253338458643013,
 'is_unbalance': True,
 'learning_rate': 0.1969438727735549,
 'low_bound': 0.0416341861513369,
 'max_bin': 169,
 'max_depth': 4,
 'n_estimators': 2445,
 'num_leaves': 255,
 'reg_alpha': 0.1561946763791145,
 'reg_lambda': 0.0236339717019237,
 'sample_weight': None,
 'subsample': 0.7463830288952731}

### Load selected features

In [20]:
from utils.feature_selection_utils import prepare_features

if "feature_num" in params:
    feature_num = params["feature_num"]
    corr_thresh = params["corr_thresh"]
    del params["feature_num"]
    del params["corr_thresh"]

fi = pd.read_csv("model/features/feature_importance.csv")
features, feature_dict = prepare_features(train_df, fi, feature_num, corr_thresh)

assert len(features) == len(set(features))

display(features, len(features))

['funding_rate',
 'weekday',
 'btcdom_high',
 'atr',
 'open',
 'high',
 'btcdom_volume',
 'rsi',
 'low',
 'close',
 'cci',
 'macdhist_prev_4',
 'btcdom_volume_prev_4',
 'low_prev_4',
 'btcd_volume_prev_24',
 'btcdom_volume_prev_24',
 'btcdom_volume_prev_28',
 'btcdom_volume_prev_32',
 'btcdom_volume_prev_36',
 'stoch_diff_prev_36',
 'btcdom_volume_prev_40',
 'btcdom_volume_prev_44',
 'close_prev_44',
 'cci_prev_56',
 'low_prev_56',
 'btcdom_volume_prev_68',
 'fng_value_prev_72',
 'btcd_volume_prev_72',
 'btcdom_volume_prev_72',
 'stoch_diff_prev_72',
 'btcdom_volume_prev_80',
 'btcdom_volume_prev_84',
 'cci_prev_84',
 'close_prev_88',
 'btcdom_volume_prev_92',
 'fng_value_prev_120',
 'stoch_slowd_prev_128',
 'btcdom_volume_prev_132',
 'btcdom_volume_prev_136',
 'btcdom_volume_prev_148',
 'rsi_prev_148',
 'btcdom_volume_prev_164',
 'fng_value_prev_168',
 'btcdom_volume_prev_188',
 'btcdom_volume_prev_192',
 'btcdom_volume_prev_196',
 'btcdom_volume_prev_208',
 'stoch_diff_prev_212',
 'b

58

### Prepare model parameters

In [21]:
# set high and low bound for model predictions
# p > high_bound -> 1, p < low_bound -> 0
if "high_bound" in params:
    high_bound = params["high_bound"]
    del params["high_bound"]
    del params["low_bound"]
low_bound = 0

# add object weights
if "sample_weight" in params:
    sample_weight = params["sample_weight"]
    train_df["weight"] = train_df["time"].astype(np.int64) / int(1e6)
    del params["sample_weight"]
else:
    sample_weight = None

if sample_weight == "cos":
    train_df["weight"] = (train_df["weight"].max() - train_df["weight"]) / (train_df["weight"].max() - train_df["weight"].min()) * np.pi / 2
    train_df["weight"] = np.cos(train_df["weight"])
    sample_weight = True
elif sample_weight == "linear":
    train_df["weight"] = (train_df["weight"].max() - train_df["weight"]) / (train_df["weight"].max() - train_df["weight"].min())
    sample_weight = True

params["objective"] = "binary"
params["verbosity"] = -1
if params["boosting_type"] != "goss":
    params["subsample_freq"] = 1
else:
    params["subsample"] = None
    params["subsample_freq"] = None
params["importance_type"] = "gain"
params["metric"] = "average_precison"

### Train process

In [22]:
import os
from datetime import datetime

from utils.model_train_utils import conf_ppv_npv_acc_score, model_train

with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

if CFG.train_test == "fold":
    model, conf_scores, conf_object_nums, oof, val_idxs = model_train(
        train_df[train_df["time"] < test_date], 
        features, 
        params, 
        sample_weight,
        n_folds=CFG.n_folds, 
        low_bound=low_bound,
        high_bound=high_bound, 
        train_test="fold",
        bybit_tickers=bybit_tickers,
        verbose=True,
    )
    y = train_df["target"][val_idxs]
    oof = oof[val_idxs]
    oof_conf_score, oof_conf_obj_num, oof_conf_obj_pct = conf_ppv_npv_acc_score(y, oof, low_bound, high_bound)

    total_profitable = round((2 * oof_conf_score - 1) * oof_conf_obj_num)

    print(80 * "=")
    print(f"Confident object score: {oof_conf_score}\n"
          f"Total number of confident objects {oof_conf_obj_num}\n"
          f"Total number of profitable objects: {total_profitable}")

    scores = [conf_object_num * (conf_score - CFG.min_precision) * 100 for conf_object_num, conf_score in zip(conf_object_nums, conf_scores)]
    print(f"Scores: {scores}")

    # save fold run results
    results_path = "model/train/train_fold_results.csv"
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    run_row = pd.DataFrame([{
        "run_datetime": datetime.now().isoformat(timespec="minutes"),
        "params": json.dumps(params),
        "high_bound": json.dumps(high_bound),
        "features": json.dumps(features),
        "n_confident_objects": oof_conf_obj_num,
        "n_profitable_objects": total_profitable,
        "scores": json.dumps(scores),
    }])
    run_row.to_csv(results_path, mode="a", header=not os.path.exists(results_path), index=False)
elif CFG.train_test == "inference":
    model, _, _, _, _ = model_train(
        train_df, 
        features, 
        params, 
        sample_weight,
        n_folds=8, 
        low_bound=low_bound,
        high_bound=high_bound, 
        train_test="inference",
        bybit_tickers=bybit_tickers,
        verbose=False,
        test_time_days=CFG.test_time_days,
    )
    joblib.dump(model, "model/lgbm.pkl")
    # save feature dictionary for further inference
    with open("model/features.json", "w") as f:
        json.dump(feature_dict, f)

Train on the last 2 years of data up to 2024-06-04 00:00:00
[100]	training's binary_logloss: 0.502822
[200]	training's binary_logloss: 0.455542
[300]	training's binary_logloss: 0.413608
[400]	training's binary_logloss: 0.378932
[500]	training's binary_logloss: 0.341454
[600]	training's binary_logloss: 0.326176
[700]	training's binary_logloss: 0.302029
[800]	training's binary_logloss: 0.283681
[900]	training's binary_logloss: 0.2609
[1000]	training's binary_logloss: 0.245111
[1100]	training's binary_logloss: 0.227058
[1200]	training's binary_logloss: 0.216798
[1300]	training's binary_logloss: 0.203653
[1400]	training's binary_logloss: 0.187654
[1500]	training's binary_logloss: 0.17894
[1600]	training's binary_logloss: 0.164914
[1700]	training's binary_logloss: 0.15014
[1800]	training's binary_logloss: 0.139428
[1900]	training's binary_logloss: 0.12951
[2000]	training's binary_logloss: 0.121032
[2100]	training's binary_logloss: 0.110884
[2200]	training's binary_logloss: 0.103217
[2300]	t

### Visualize train results

In [23]:
if CFG.train_test == "fold":
    sns.lineplot(x=list(range(len(conf_scores))), y=conf_scores, linewidth=2)

    plt.title("Model score by folds")
    plt.xlabel("Folds")
    plt.xticks(fontsize=12)
    plt.ylabel("Score")
    plt.yticks(fontsize=12)

    plt.show()

### Display PR curve for fold predictions

In [24]:
from sklearn.metrics import PrecisionRecallDisplay

if CFG.train_test == "fold":
    disp = PrecisionRecallDisplay.from_predictions(
        y.values, oof, name="PR AUC"
    )
    plt.legend(loc="upper right")
    _ = disp.ax_.set_title("2-class Precision-Recall curve")
    disp.ax_.lines[0].set_linewidth(2)

# vol 1e6 AP=0.61

### Find the best threshold for fold predictions

In [25]:
from icecream import ic

if CFG.train_test == "fold":
    figsize = (10, 5)
    plt.figure(figsize=figsize)
    
    score_list = list()
    obj_num_list = list()
    obj_pct_list = list()
    obj_profit_list = list()
    max_obj_profit = 0
    for hb in np.arange(0.41, 0.61, 0.001):
        score, obj_num, obj_pct = conf_ppv_npv_acc_score(y.reset_index(drop=True), oof, 0, hb)
        if score == 0:
            obj_num = 0
            obj_pct = 0
        bound, score, obj_num, obj_pct = round(hb, 4), round(score, 5), round(obj_num, 2), round(obj_pct, 2)
        obj_profit = round((2 * score - 1) * obj_num)
        score_list.append(score)
        obj_num_list.append(obj_num)
        obj_pct_list.append(obj_pct)
        obj_profit_list.append(obj_profit)
        max_obj_profit = max(max_obj_profit, obj_profit)
        ic(bound, score, obj_num, obj_pct, obj_profit)

    obj_profit_list = [o / max_obj_profit for o in obj_profit_list]
    line1 = plt.plot(np.arange(0.41, 0.61, 0.001), score_list, label="precison score", linewidth=2)
    line2 = plt.plot(np.arange(0.41, 0.61, 0.001), obj_pct_list, label="object pct", linewidth=2)
    line3 = plt.plot(np.arange(0.41, 0.61, 0.001), obj_profit_list, label="number of profit objects", linewidth=2)
    
    plt.legend()
    plt.xlabel("Threshold")
    plt.show()

### Model feature importance

In [26]:
if CFG.train_test == "fold":
    lgb.plot_importance(model, importance_type="gain", figsize=(5, 8), title="LightGBM Feature Importance (Gain)")
    plt.show()

# Error analysis

### Distribution of model pseudo-residuals

In [27]:
from utils.model_test_utils import logloss

if CFG.train_test == "fold":
    y_true = y.values.reshape(-1, 1)
    residuals = -logloss(y_true, oof)

    sns.displot(np.log1p(residuals), bins=100)

    plt.xlabel("Model prediction residuals")
    plt.xticks(fontsize=10)
    plt.ylabel("Number of predictions")
    plt.yticks(fontsize=10)

In [28]:
if CFG.train_test == "fold":
    prob_df = pd.DataFrame({"pred": oof.squeeze(), "target": y})

    # common_norm=False normalizes each class separately so the imbalanced
    # classes are comparable and their separation is visible
    sns.histplot(
        data=prob_df, x="pred", hue="target",
        bins=100, stat="density", common_norm=False,
        element="step", alpha=0.4,
    )

    plt.title("Predicted probability distribution by class")
    plt.xlabel("Predicted probability")
    plt.xticks(fontsize=10)
    plt.ylabel("Density")
    plt.yticks(fontsize=10)
    plt.show()


### Calibration curve

In [29]:
from sklearn.calibration import CalibrationDisplay

if CFG.train_test == "fold":
    # strategy="quantile" -> equal-count bins, since the probabilities
    # are concentrated in the mid-range rather than spread evenly
    disp = CalibrationDisplay.from_predictions(
        y, oof.squeeze(), n_bins=10, strategy="quantile", name="LGBM",
    )
    disp.ax_.set_title("Calibration curve")
    disp.ax_.lines[0].set_linewidth(2)
    plt.show()

### Show worst predictions

In [30]:
if CFG.train_test == "fold":
    prob_df["residuals"] = np.abs(prob_df["target"] - prob_df["pred"])
    worst_idxs = prob_df.query("pred >= @high_bound").sort_values("residuals", ascending=False).head(20).index
    worst_idxs_df = train_df.loc[worst_idxs, ["time", "ticker", "target", "ttype"]]
    worst_idxs_df["pred"] = prob_df.query("pred >= @high_bound").sort_values("residuals", ascending=False).head(20)["pred"]
    display(worst_idxs_df)

### Show best predictions

In [31]:
if CFG.train_test == "fold":
    best_idxs = prob_df.query("pred >= @high_bound").sort_values("residuals").head(20).index
    best_idxs_df = train_df.loc[best_idxs, ["time", "ticker", "target", "ttype"]]
    best_idxs_df["pred"] = prob_df.query("pred >= @high_bound").sort_values("residuals").head(20)["pred"]
    display(best_idxs_df)

# Compare indicator / signal values for bot and optimizer

In [32]:
# import fireducks.pandas as pd
# from signals.find_signal import SignalFactory

# ttype = "sell"
# ticker = "BADGERUSDT"
# month = 7
# day = 15
# hour = 23
# configs = ConfigFactory.factory(environ).configs

# x = pd.read_csv(f"../bot/ticker_dataframes/{ticker}_1h_{ttype}_{month}_{day}_{hour}.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
# y = pd.read_csv(f"../bot/ticker_dataframes/{ticker}_4h_{ttype}_{month}_{day}_{hour}.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)

# # add Volume24
# vol24 = indicators.Volume24(ttype, configs)
# x = vol24.get_indicator(x, "", "1h", 0)
# # add Pattern
# pattern = indicators.Pattern(ttype, configs)
# x = pattern.get_indicator(x, "", "", 0)
# # add trend
# trend = indicators.Trend(ttype, configs)
# y = trend.get_indicator(y, "", "", 0)

# # cols = ["time", "open", "high", "low", "close", "volume", "rsi", "stoch_slowk", "stoch_slowd", "linear_reg", "linear_reg_angle", "macd", "macdsignal", "macdhist"]
# cols = ["time", "open", "high", "low", "close", "volume", "linear_reg", "linear_reg_angle", "high_max", "low_min", "volume_24"]

# higher_features = ["time_4h", "linear_reg", "linear_reg_angle", "macd", "macdhist",  "macd_dir", 
#                    "macdsignal", "macdsignal_dir"]
# x["time"] = pd.to_datetime(x["time"])
# y["time"] = pd.to_datetime(y["time"])
# y["time_4h"] = y["time"] + pd.to_timedelta(3, unit="h")
# x[["time"] + higher_features] = pd.merge(x[["time"]], y[higher_features], how="left", left_on="time", right_on="time_4h")

# # x = x.drop(columns=["time_4h"])
# # y = y.drop(columns=["time_4h"])
# x = x.ffill()
# x = x.reset_index(drop=True)

# # get Swing pattern
# pattern = SignalFactory().factory("Pattern", ttype, configs)
# pattern_points = pattern.find_signal(x)
# trend = SignalFactory().factory("Trend", ttype, configs)
# trend_points = trend.find_signal(x)
# idxs = np.where((pattern_points > 0) & (trend_points > 0))
# display(x.loc[idxs[0], cols])

# z = pd.read_pickle(f"data/tickers/{ticker}_1h.pkl")
# v = pd.read_pickle(f"data/tickers/{ticker}_4h.pkl")

# # add Volume24
# vol24 = indicators.Volume24(ttype, configs)
# z = vol24.get_indicator(z, "", "1h", 0)
# # add Pattern
# pattern = indicators.Pattern(ttype, configs)
# z = pattern.get_indicator(z, "", "", 0)
# # add Trend
# trend = indicators.Trend(ttype, configs)
# v = trend.get_indicator(v, "", "", 0)
# z.tail(48)

# v["time_4h"] = v["time"] + pd.to_timedelta(3, unit="h")
# z[["time"] + higher_features] = pd.merge(z[["time"]], y[higher_features], how="left", left_on="time", right_on="time_4h")

# z = z.drop(columns=["time_4h"])
# v = v.drop(columns=["time_4h"])
# z = z.ffill()
# z = z.reset_index(drop=True)

# # get Swing pattern
# pattern = SignalFactory().factory("Pattern", ttype, configs)
# pattern_points = pattern.find_signal(z)
# trend = SignalFactory().factory("Trend", ttype, configs)
# trend_points = trend.find_signal(z)
# idxs = np.where((pattern_points > 0) & (trend_points > 0))
# display(z.loc[idxs[0], cols])